# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name: {}".format(metadata.name))
print("Description: {}".format(metadata.description))
print("Version: {}".format(metadata.version))
print("License: {}".format(metadata.license))
print("Published: {}".format(metadata.datePublished))

## 2. Data Overview
Review available record sets, fields, and their IDs. All Croissant entities should be referenced via their `@id` field.

First list the available record sets, then for each record set list fields and columns.

In [ ]:
# List all record sets in the dataset and their @id
print("Available Record Sets (@id list):")
record_set_ids = []
for record_set in dataset.metadata.recordSets:
    print(f"- {record_set['@id']}: name={record_set.get('name','N/A')}, columns={[col['@id'] for col in record_set.get('columns',[])]}")
    record_set_ids.append(record_set['@id'])

if not record_set_ids:
    print("No record sets found in dataset metadata. The dataset might have one main data file.")

In [ ]:
# If record sets are empty (FAIR^2 dataset uses one main table/file), use the utility to list available data file objects
data_files = []
if not record_set_ids:
    # Try to list file objects (which often act as record sets for simple datasets)
    if hasattr(metadata, 'fileObjects'):
        for file_obj in metadata.fileObjects:
            print(f"FileObject: @id={file_obj['@id']}, name={file_obj.get('name','')}, encoding={file_obj.get('encodingFormat','')}, url={file_obj.get('contentUrl','')}")
            data_files.append(file_obj['@id'])
        record_set_ids = data_files
    else:
        print("No file objects found. Please inspect dataset.metadata for available data sources.")

Next, for each record set or data table, print the fields/columns with their `@id`, name, and data type where available.

In [ ]:
# Show columns/fields for a given record set. Use the discovered record_set_ids
for record_set_id in record_set_ids:
    print(f"\nColumns in record_set/@id = {record_set_id}:")
    try:
        for record_set in dataset.metadata.recordSets:
            if record_set['@id'] == record_set_id:
                for col in record_set.get('columns',[]):
                    print(f"  - @id={col['@id']}, name={col.get('name','')}, dataType={col.get('dataType','N/A')}")
    except Exception as e:
        print("Could not fetch columns for record set", record_set_id, str(e))

## 3. Data Extraction
Load data from a specific record set or data table into a DataFrame for analysis. Use the record set and field `@id`s found above.

In [ ]:
# Extract data from all discovered record sets (or file objects/tables)
# These will become DataFrames, stored in dataframes dict by their @id
dataframes = dict()

# If record set IDs are available
for record_set_id in record_set_ids:
    print(f"\nLoading DataFrame for record_set_id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Fields: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head(3))
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric fields, or grouping data.
We'll demonstrate this for the first available record set/dataframe. 

In [ ]:
import numpy as np

# Use the first dataframe (if available)
if dataframes:
    first_record_set_id = next(iter(dataframes.keys()))
    df = dataframes[first_record_set_id]

    print(f"Selecting DataFrame: {first_record_set_id}")
    print("Columns:", df.columns.tolist())

    # Try to select a numeric field automatically (common names or by dtype)
    candidate_numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]]
    if not candidate_numeric_fields:
        # Try to cast columns to float to detect numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except:
                continue
        candidate_numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]]

    print("Numeric fields detected:", candidate_numeric_fields)

    if candidate_numeric_fields:
        numeric_field = candidate_numeric_fields[0]  # Just as example
        threshold = df[numeric_field].mean() if np.isfinite(df[numeric_field].mean()) else 0

        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} rows")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to select a group field: pick string/object with <20 unique vals
        candidate_group_fields = [col for col in df.columns if df[col].dtype==object and df[col].nunique() < 20]
        print("Candidate group fields:", candidate_group_fields)
        if candidate_group_fields:
            group_field = candidate_group_fields[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field} (showing means):")
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA. Please review columns above.")
else:
    print("No DataFrames to analyze. Please check previous steps.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll plot a histogram of the first numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and candidate_numeric_fields:
    field_to_plot = candidate_numeric_fields[0]
    plt.figure(figsize=(8,4))
    sns.histplot(df[field_to_plot].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of '{field_to_plot}' in record_set: {first_record_set_id}")
    plt.xlabel(field_to_plot)
    plt.show()
else:
    print("No suitable numeric field for visualization.")

## 6. Conclusion
This notebook used the `mlcroissant` library to:
- Load FAIR² dataset metadata and records with all references by their `@id`
- Overview its record sets, fields, and data structure
- Extract data as DataFrames and run basic filtering, normalization, and grouping
- Visualize a numeric field's distribution

**Key findings and next steps:**
- Review the EDA and groupings for outliers or trends.
- Use the provided columns for advanced analyses, e.g., protein expression comparisons.
- Refer to all Croissant schema objects by their `@id` for reproducibility & clarity.
